# Sprint 5 — LightGBM, XGBoost y Optimización de Hiperparámetros
## Proyecto: Productividad Asesores de Negocios

---

### Objetivo del sprint

Entrenar LightGBM y XGBoost con optimización de hiperparámetros (Optuna)
y comparar sistemáticamente contra el baseline de Logistic Regression.

### Piso de métricas a superar (Sprint 4)

| Métrica | Logistic Regression | Objetivo mínimo |
|---------|--------------------|-----------------|
| Macro F1 | 0.5696 | > 0.60 (+3pp) |
| Kappa | 0.4248 | > 0.45 |
| AUC OvR | 0.8079 | > 0.83 |

### ⚠️ Advertencia metodológica

> Con ~1,600 observaciones en train, LightGBM puede overfit fácilmente.
> Se usa k-fold estratificado (k=5) en Optuna para evaluar cada trial.
> Un modelo con F1_cv >> F1_test indica overfitting — reportar ambos.


## 1. Configuración y carga de datos

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import mlflow
import mlflow.lightgbm
import mlflow.xgboost
import optuna
import lightgbm as lgb
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    f1_score, cohen_kappa_score, roc_auc_score,
    accuracy_score, classification_report, ConfusionMatrixDisplay,
    confusion_matrix,
)
from sklearn.metrics import make_scorer

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

PROCESSED  = Path('../data/processed')
FIG_DIR    = Path('../reports/figures')
MLRUNS_DIR = Path('../mlruns')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Cargar datos procesados
data = np.load(PROCESSED / 'features_processed.npz', allow_pickle=True)
X_train = data['X_train']
X_test  = data['X_test']
y_train = data['y_train']
y_test  = data['y_test']

label_encoder = joblib.load(PROCESSED / 'label_encoder.joblib')
CLASS_NAMES   = list(label_encoder.classes_)
N_CLASSES     = len(CLASS_NAMES)

# MLflow con backend SQLite
DB_PATH_MLFLOW = Path('../mlruns/mlflow.db')
mlflow.set_tracking_uri(f'sqlite:///{DB_PATH_MLFLOW.resolve()}')
mlflow.set_experiment('advisor_risk_classification')

print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'Clases : {CLASS_NAMES}')
print(f'MLflow : {mlflow.get_tracking_uri()}')


X_train: (1603, 22)
X_test : (401, 22)
Clases : ['Q1_BAJO', 'Q2_MEDIO_BAJO', 'Q3_MEDIO_ALTO', 'Q4_ALTO']
MLflow : sqlite:///C:\Users\Marin\Documents\PROYECTO ML_OPS\productividad-asesores-negocios\mlruns\mlflow.db


## 2. Funciones de evaluación compartidas

In [2]:
def calcular_metricas(y_true, y_pred, y_prob, class_names):
    '''
    Calcula metricas completas de evaluacion multiclase.
    Metrica principal: macro F1.
    '''
    metricas = {
        'macro_f1' : f1_score(y_true, y_pred, average='macro'),
        'kappa'    : cohen_kappa_score(y_true, y_pred),
        'auc_ovr'  : roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro'),
        'accuracy' : accuracy_score(y_true, y_pred),
    }
    f1_por_clase = f1_score(y_true, y_pred, average=None)
    for i, cls in enumerate(class_names):
        metricas[f'f1_{cls}'] = f1_por_clase[i]
    return metricas


def plot_confusion_matrix(y_true, y_pred, class_names, titulo, filepath):
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='.2f')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()
    return filepath


# Scorer para Optuna
cv_scorer = make_scorer(f1_score, average='macro')
cv        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Funciones de evaluacion definidas.')


Funciones de evaluacion definidas.


## 3. LightGBM — Optimización con Optuna

Optuna busca los mejores hiperparámetros minimizando el macro F1 negativo
en validación cruzada estratificada (k=5).

Se ejecutan **50 trials** — balance entre tiempo de cómputo y cobertura
del espacio de hiperparámetros con ~1,600 observaciones.


In [3]:
def objective_lgbm(trial):
    '''
    Funcion objetivo para Optuna — LightGBM.
    Maximiza macro F1 en validacion cruzada estratificada.
    '''
    params = {
        'objective'        : 'multiclass',
        'num_class'        : N_CLASSES,
        'metric'           : 'multi_logloss',
        'verbosity'        : -1,
        'random_state'     : 42,
        'is_unbalance'     : True,   # Manejo de desbalance de clases
        # Hiperparametros a optimizar
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 800),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 15, 63),
        'max_depth'        : trial.suggest_int('max_depth', 3, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring=cv_scorer, n_jobs=-1
    )
    return scores.mean()


print('Ejecutando optimizacion Optuna para LightGBM (50 trials)...')
print('Esto puede tardar 3-5 minutos...')

study_lgbm = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f'\nMejor macro F1 CV (LightGBM): {study_lgbm.best_value:.4f}')
print(f'Mejores hiperparametros:')
for k, v in study_lgbm.best_params.items():
    print(f'  {k:<25}: {v}')


Ejecutando optimizacion Optuna para LightGBM (50 trials)...
Esto puede tardar 3-5 minutos...


  0%|          | 0/50 [00:00<?, ?it/s]


Mejor macro F1 CV (LightGBM): 0.6138
Mejores hiperparametros:
  n_estimators             : 175
  learning_rate            : 0.13625252988274353
  num_leaves               : 26
  max_depth                : 3
  min_child_samples        : 32
  subsample                : 0.8546743841357609
  colsample_bytree         : 0.9581541304120595
  reg_alpha                : 0.0011949356785869657
  reg_lambda               : 0.047781950780058036


## 4. LightGBM — Entrenamiento final y registro en MLflow

In [4]:
# Entrenar LightGBM con los mejores hiperparametros
best_lgbm = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=N_CLASSES,
    verbosity=-1,
    random_state=42,
    is_unbalance=True,
    **study_lgbm.best_params
)

with mlflow.start_run(run_name='lightgbm_optuna') as run_lgbm:

    # Registrar hiperparametros
    mlflow.log_params({
        'model_type'   : 'LightGBM',
        'is_unbalance' : True,
        'optuna_trials': 50,
        'cv_folds'     : 5,
        'target'       : 'TARGET_B',
        **study_lgbm.best_params
    })
    mlflow.log_metric('cv_macro_f1_best', study_lgbm.best_value)

    # Entrenar sobre train completo
    best_lgbm.fit(X_train, y_train)

    # Evaluar en test
    y_pred_lgbm = best_lgbm.predict(X_test)
    y_prob_lgbm = best_lgbm.predict_proba(X_test)

    metricas_lgbm = calcular_metricas(y_test, y_pred_lgbm, y_prob_lgbm, CLASS_NAMES)
    mlflow.log_metrics({f'test_{k}': v for k, v in metricas_lgbm.items()})

    # Matriz de confusion
    cm_path_lgbm = FIG_DIR / '05_confusion_matrix_lgbm.png'
    plot_confusion_matrix(
        y_test, y_pred_lgbm, CLASS_NAMES,
        'LightGBM — Matriz de Confusion (normalizada)', cm_path_lgbm
    )
    mlflow.log_artifact(str(cm_path_lgbm))
    mlflow.lightgbm.log_model(best_lgbm, 'lightgbm_model')

    run_id_lgbm = run_lgbm.info.run_id

print('=== METRICAS EN TEST — LIGHTGBM ===')
print(f'  CV Macro F1 (Optuna best) : {study_lgbm.best_value:.4f}')
print(f'  Test Macro F1             : {metricas_lgbm["macro_f1"]:.4f}  <- METRICA PRINCIPAL')
print(f'  Test Kappa                : {metricas_lgbm["kappa"]:.4f}')
print(f'  Test AUC OvR              : {metricas_lgbm["auc_ovr"]:.4f}')
print(f'  Test Accuracy             : {metricas_lgbm["accuracy"]:.4f}')
print(f'\nF1 por clase:')
for cls in CLASS_NAMES:
    print(f'  {cls:<20}: {metricas_lgbm[f"f1_{cls}"]:.4f}')
print(f'\nMLflow run_id: {run_id_lgbm}')


2026/06/04 18:49:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 18:49:05 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/04 18:49:25 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


=== METRICAS EN TEST — LIGHTGBM ===
  CV Macro F1 (Optuna best) : 0.6138
  Test Macro F1             : 0.6144  <- METRICA PRINCIPAL
  Test Kappa                : 0.4813
  Test AUC OvR              : 0.8380
  Test Accuracy             : 0.6110

F1 por clase:
  Q1_BAJO             : 0.7292
  Q2_MEDIO_BAJO       : 0.5000
  Q3_MEDIO_ALTO       : 0.4670
  Q4_ALTO             : 0.7614

MLflow run_id: 564f0d81bf364d83bbce561dbacf91ec


## 5. XGBoost — Optimización con Optuna

Mismo proceso que LightGBM para comparación justa.
XGBoost usa `sample_weight` para manejo de desbalance
en lugar del parámetro `is_unbalance` de LightGBM.


In [5]:
# Calcular pesos por clase para XGBoost
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

def objective_xgb(trial):
    '''
    Funcion objetivo para Optuna — XGBoost.
    Usa sample_weight para manejo de desbalance.
    '''
    params = {
        'objective'        : 'multi:softprob',
        'num_class'        : N_CLASSES,
        'eval_metric'      : 'mlogloss',
        'verbosity'        : 0,
        'random_state'     : 42,
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 800),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 3, 8),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'gamma'            : trial.suggest_float('gamma', 0, 5),
    }
    model = xgb.XGBClassifier(**params, use_label_encoder=False)

    # CV manual para pasar sample_weight
    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        sw_tr       = sample_weights[train_idx]
        model.fit(X_tr, y_tr, sample_weight=sw_tr, verbose=False)
        y_pred_val = model.predict(X_val)
        scores.append(f1_score(y_val, y_pred_val, average='macro'))
    return np.mean(scores)


print('Ejecutando optimizacion Optuna para XGBoost (50 trials)...')
print('Esto puede tardar 5-8 minutos...')

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

print(f'\nMejor macro F1 CV (XGBoost): {study_xgb.best_value:.4f}')
print(f'Mejores hiperparametros:')
for k, v in study_xgb.best_params.items():
    print(f'  {k:<25}: {v}')


Ejecutando optimizacion Optuna para XGBoost (50 trials)...
Esto puede tardar 5-8 minutos...


  0%|          | 0/50 [00:00<?, ?it/s]


Mejor macro F1 CV (XGBoost): 0.6133
Mejores hiperparametros:
  n_estimators             : 482
  learning_rate            : 0.08295709277032152
  max_depth                : 7
  min_child_weight         : 9
  subsample                : 0.6656303620724494
  colsample_bytree         : 0.653045169342304
  reg_alpha                : 0.3784017226809909
  reg_lambda               : 0.006080819983119863
  gamma                    : 3.7378657717091


## 6. XGBoost — Entrenamiento final y registro en MLflow

In [6]:
best_xgb = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=N_CLASSES,
    verbosity=0,
    random_state=42,
    use_label_encoder=False,
    **study_xgb.best_params
)

with mlflow.start_run(run_name='xgboost_optuna') as run_xgb:

    mlflow.log_params({
        'model_type'      : 'XGBoost',
        'class_weighting' : 'sample_weight_balanced',
        'optuna_trials'   : 50,
        'cv_folds'        : 5,
        'target'          : 'TARGET_B',
        **study_xgb.best_params
    })
    mlflow.log_metric('cv_macro_f1_best', study_xgb.best_value)

    best_xgb.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred_xgb = best_xgb.predict(X_test)
    y_prob_xgb = best_xgb.predict_proba(X_test)

    metricas_xgb = calcular_metricas(y_test, y_pred_xgb, y_prob_xgb, CLASS_NAMES)
    mlflow.log_metrics({f'test_{k}': v for k, v in metricas_xgb.items()})

    cm_path_xgb = FIG_DIR / '06_confusion_matrix_xgb.png'
    plot_confusion_matrix(
        y_test, y_pred_xgb, CLASS_NAMES,
        'XGBoost — Matriz de Confusion (normalizada)', cm_path_xgb
    )
    mlflow.log_artifact(str(cm_path_xgb))
    mlflow.xgboost.log_model(best_xgb, 'xgboost_model')

    run_id_xgb = run_xgb.info.run_id

print('=== METRICAS EN TEST — XGBOOST ===')
print(f'  CV Macro F1 (Optuna best) : {study_xgb.best_value:.4f}')
print(f'  Test Macro F1             : {metricas_xgb["macro_f1"]:.4f}  <- METRICA PRINCIPAL')
print(f'  Test Kappa                : {metricas_xgb["kappa"]:.4f}')
print(f'  Test AUC OvR              : {metricas_xgb["auc_ovr"]:.4f}')
print(f'  Test Accuracy             : {metricas_xgb["accuracy"]:.4f}')
print(f'\nF1 por clase:')
for cls in CLASS_NAMES:
    print(f'  {cls:<20}: {metricas_xgb[f"f1_{cls}"]:.4f}')


2026/06/04 18:54:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 18:54:52 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


=== METRICAS EN TEST — XGBOOST ===
  CV Macro F1 (Optuna best) : 0.6133
  Test Macro F1             : 0.5700  <- METRICA PRINCIPAL
  Test Kappa                : 0.4282
  Test AUC OvR              : 0.8303
  Test Accuracy             : 0.5711

F1 por clase:
  Q1_BAJO             : 0.6667
  Q2_MEDIO_BAJO       : 0.4486
  Q3_MEDIO_ALTO       : 0.4278
  Q4_ALTO             : 0.7368


## 7. Tabla comparativa — los 3 modelos

Comparación final para seleccionar el modelo principal del Sprint 6.


In [9]:
# Metricas del baseline LR (Sprint 4 — valores conocidos)
metricas_lr = {
    'macro_f1': 0.5696,
    'kappa'   : 0.4248,
    'auc_ovr' : 0.8079,
    'accuracy': 0.5686,
}

print('=== TABLA COMPARATIVA — 3 MODELOS ===')
header = f"{'Metrica':<20} {'LR Baseline':>14} {'LightGBM':>14} {'XGBoost':>14}"
print(header)
print('─' * 65)

for metrica in ['macro_f1', 'kappa', 'auc_ovr', 'accuracy']:
    lr_val   = metricas_lr[metrica]
    lgbm_val = metricas_lgbm[metrica]
    xgb_val  = metricas_xgb[metrica]
    mejor    = max(lr_val, lgbm_val, xgb_val)
    lr_str   = f'{lr_val:.4f}' + (' *' if lr_val == mejor else '  ')
    lgbm_str = f'{lgbm_val:.4f}' + (' *' if lgbm_val == mejor else '  ')
    xgb_str  = f'{xgb_val:.4f}' + (' *' if xgb_val == mejor else '  ')
    fila = f'{metrica:<20} {lr_str:>14} {lgbm_str:>14} {xgb_str:>14}'
    print(fila)

print('─' * 65)
print('* = mejor en esa metrica')

# Seleccion del modelo final
mejor_f1_lgbm = metricas_lgbm['macro_f1']
mejor_f1_xgb  = metricas_xgb['macro_f1']

if mejor_f1_lgbm >= mejor_f1_xgb:
    modelo_final = 'LightGBM'
    modelo_obj   = best_lgbm
    metricas_fin = metricas_lgbm
else:
    modelo_final = 'XGBoost'
    modelo_obj   = best_xgb
    metricas_fin = metricas_xgb

mejora = metricas_fin['macro_f1'] - metricas_lr['macro_f1']
print(f'\nMODELO SELECCIONADO PARA SPRINT 6: {modelo_final}')
print(f'  Macro F1 test  : {metricas_fin["macro_f1"]:.4f}')
print(f'  Mejora baseline: +{mejora:.4f} ({mejora*100:.1f}pp)')

# Guardar modelo final
joblib.dump(modelo_obj, PROCESSED / 'modelo_final.joblib')
print(f'\nModelo guardado en data/processed/modelo_final.joblib')

=== TABLA COMPARATIVA — 3 MODELOS ===
Metrica                 LR Baseline       LightGBM        XGBoost
─────────────────────────────────────────────────────────────────
macro_f1                   0.5696         0.6144 *       0.5700  
kappa                      0.4248         0.4813 *       0.4282  
auc_ovr                    0.8079         0.8380 *       0.8303  
accuracy                   0.5686         0.6110 *       0.5711  
─────────────────────────────────────────────────────────────────
* = mejor en esa metrica

MODELO SELECCIONADO PARA SPRINT 6: LightGBM
  Macro F1 test  : 0.6144
  Mejora baseline: +0.0448 (4.5pp)

Modelo guardado en data/processed/modelo_final.joblib
